# Y1 一次性训练数据生产

## tl;dr

本 Notebook 是 `02_数据处理` 的唯一生产入口。它从 `data.z` 独立读取原始数据，
一次生成版本化的 Train、Valid、Test 多视图数据；`03_模型训练` 只加载固定视图，
不再补值、编码、标准化或构造特征。

生产完成的唯一可信标志是 `processed_data_v1/READY`。没有该文件时，模型必须拒绝训练。

### Key Assumptions

- 所有映射、频率、分位数和标准化参数只在 Train `[486, 2918)` 拟合。
- 当前 `mask_x=False` 不生成监督样本；历史缺口只使用预测时点以前的数据补值。
- 真实观测值不做截尾；只有趋势补充值受 Train 分位数边界保护。
- Train 与 Valid 合并重训时仍使用原 Train 拟合的处理参数。
- 固定随机种子为 42，行顺序为时间升序、股票编号升序。

## Context & Methods

### 1. 参数、目录与固定特征清单

In [1]:
from __future__ import annotations

import gc
import hashlib
import json
import math
import mmap
import os
import shutil
import struct
import tempfile
import time
from pathlib import Path

import numpy as np
import pandas as pd
import zstandard as zstd
from scipy.stats import rankdata

BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name != "02_数据处理":
    raise RuntimeError(f"请从 02_数据处理 目录启动；当前目录为 {BASE_DIR}")
ROOT_DIR = BASE_DIR.parent
SOURCE_PATH = ROOT_DIR / "data.z"
OUTPUT_DIR = BASE_DIR / "processed_data_v1"
BUILD_CACHE_DIR = BASE_DIR / ".build_cache"
READY_PATH = OUTPUT_DIR / "READY"
MANIFEST_PATH = OUTPUT_DIR / "manifest.json"

T, S = 3603, 5282
NUMERIC_FEATURE_COUNT, CATEGORY_FEATURE_COUNT = 99, 9
TRAIN_START, VALID_START, TEST_START = 486, 2918, 3161
SEED = 42
LAGS = (1, 5, 20, 60)
WINDOWS = (5, 20, 60)
LOW_CARDINALITY_CATEGORIES = (0, 1, 2, 3, 4, 6, 7, 8)
HIGH_CARDINALITY_CATEGORY = 5
EXPECTED_ROWS = {"train": 6_489_099, "valid": 982_972, "test": 2_042_538}
SPLITS = {
    "train": (TRAIN_START, VALID_START, True),
    "valid": (VALID_START, TEST_START, True),
    "test": (TEST_START, T, False),
}

# 由历史最佳实验冻结到数据处理层；不再运行时读取 03_模型训练。
SELECTED_NUMERIC = np.asarray([
    8, 11, 57, 41, 90, 68, 39, 40, 73, 47,
    53, 72, 48, 74, 86, 38, 50, 3, 42, 71,
    87, 49, 4, 55, 85, 75, 56, 76, 51, 67,
    88, 61, 79, 1, 91, 84, 15, 60, 64, 43,
], dtype=np.int32)
HISTORY_NUMERIC = SELECTED_NUMERIC[:20].copy()

BUILD_REQUIRED = not READY_PATH.exists()
print({
    "source": str(SOURCE_PATH),
    "output": str(OUTPUT_DIR),
    "build_required": BUILD_REQUIRED,
    "selected_numeric": SELECTED_NUMERIC.tolist(),
    "history_numeric": HISTORY_NUMERIC.tolist(),
})

{'source': 'D:\\google_dl\\book\\友安杯\\data.z', 'output': 'D:\\google_dl\\book\\友安杯\\02_数据处理\\processed_data_v1', 'build_required': False, 'selected_numeric': [8, 11, 57, 41, 90, 68, 39, 40, 73, 47, 53, 72, 48, 74, 86, 38, 50, 3, 42, 71, 87, 49, 4, 55, 85, 75, 56, 76, 51, 67, 88, 61, 79, 1, 91, 84, 15, 60, 64, 43], 'history_numeric': [8, 11, 57, 41, 90, 68, 39, 40, 73, 47, 53, 72, 48, 74, 86, 38, 50, 3, 42, 71]}


### 2. 独立读取 `data.z`

In [2]:
class LimitedReader:
    def __init__(self, raw, remaining: int):
        self.raw = raw
        self.remaining = int(remaining)

    def readable(self):
        return True

    def read(self, size: int = -1):
        if self.remaining <= 0:
            return b""
        if size is None or size < 0:
            size = self.remaining
        size = min(int(size), self.remaining)
        chunk = self.raw.read(size)
        self.remaining -= len(chunk)
        return chunk


def ensure_source_cache() -> Path:
    BUILD_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    output_path = BUILD_CACHE_DIR / "payload.pkl"
    expected_size = 9_096_840_663
    if output_path.exists() and output_path.stat().st_size == expected_size:
        return output_path
    temporary_path = BUILD_CACHE_DIR / "payload.pkl.partial"
    if temporary_path.exists():
        temporary_path.unlink()
    print("正在流式解压 data.z（约 9.1 GB 临时缓存）...")
    with SOURCE_PATH.open("rb") as source:
        header = source.read(7)
        if len(header) != 7 or header[:3] != b"\x80\x05\x42":
            raise ValueError("data.z 外层格式不符合预期")
        payload_length = struct.unpack("<I", header[3:7])[0]
        limited = LimitedReader(source, payload_length)
        decompressor = zstd.ZstdDecompressor()
        with decompressor.stream_reader(limited) as reader, temporary_path.open("wb") as target:
            shutil.copyfileobj(reader, target, length=16 * 1024 * 1024)
    if temporary_path.stat().st_size != expected_size:
        raise ValueError(f"解压大小异常：{temporary_path.stat().st_size}")
    temporary_path.replace(output_path)
    return output_path


def locate_raw_array(mm: mmap.mmap, key: bytes, byte_length: int) -> int:
    key_pattern = bytes([0x8C, len(key)]) + key + b"\x94"
    if byte_length <= np.iinfo(np.uint32).max:
        data_pattern = b"\x42" + struct.pack("<I", byte_length)
    else:
        data_pattern = b"\x8E" + struct.pack("<Q", byte_length)
    search_from = 0
    while True:
        key_position = mm.find(key_pattern, search_from)
        if key_position < 0:
            break
        opcode_position = mm.find(data_pattern, key_position, key_position + 512)
        if opcode_position >= 0:
            return opcode_position + len(data_pattern)
        search_from = key_position + 1
    raise ValueError(f"无法定位数组 {key!r}")


class CompetitionData:
    def __init__(self, pickle_path: Path):
        self.pickle_path = pickle_path
        n = T * S
        with pickle_path.open("rb") as handle:
            mm = mmap.mmap(handle.fileno(), 0, access=mmap.ACCESS_READ)
            offsets = {
                "num_x": locate_raw_array(mm, b"num_x", n * NUMERIC_FEATURE_COUNT * 4),
                "cat_x": locate_raw_array(mm, b"cat_x", n * CATEGORY_FEATURE_COUNT * 8),
                "y1": locate_raw_array(mm, b"y1", n * 4),
                "mask_x": locate_raw_array(mm, b"mask_x", n),
                "mask_y": locate_raw_array(mm, b"mask_y", n),
            }
            mm.close()
        self.num_x = np.memmap(pickle_path, np.float32, "r", offsets["num_x"], (T, S, 99))
        self.cat_x = np.memmap(pickle_path, np.int64, "r", offsets["cat_x"], (T, S, 9))
        self.y1 = np.memmap(pickle_path, np.float32, "r", offsets["y1"], (T, S))
        self.mask_x = np.memmap(pickle_path, np.bool_, "r", offsets["mask_x"], (T, S))
        self.mask_y = np.memmap(pickle_path, np.bool_, "r", offsets["mask_y"], (T, S))
        self.first_valid_x = np.argmax(self.mask_x, axis=0).astype(np.int32)
        self.validate()

    def validate(self):
        assert self.num_x.shape == (T, S, 99)
        assert self.cat_x.shape == (T, S, 9)
        assert self.y1.shape == self.mask_x.shape == self.mask_y.shape == (T, S)
        assert not np.any(self.mask_y[TRAIN_START:TEST_START] & ~self.mask_x[TRAIN_START:TEST_START])
        assert not np.any(np.isfinite(self.y1[TEST_START:]))


data = None
if BUILD_REQUIRED:
    source_cache_path = ensure_source_cache()
    data = CompetitionData(source_cache_path)
    print({
        "num_x": data.num_x.shape,
        "cat_x": data.cat_x.shape,
        "mask_x_ratio": float(np.asarray(data.mask_x).mean()),
    })
else:
    print("READY 已存在：跳过原始数据解压和全量重建。")

READY 已存在：跳过原始数据解压和全量重建。


## Data

### 3. 固定特征布局与原子写入工具

In [3]:
raw_names = [f"num_{i}" for i in SELECTED_NUMERIC]
rank_names = [f"rank_num_{i}" for i in HISTORY_NUMERIC]
lag_names = [f"lag_{lag}_num_{i}" for lag in LAGS for i in HISTORY_NUMERIC]
rolling_names = [
    f"roll_{stat}_{window}_num_{i}"
    for window in WINDOWS for stat in ("mean", "std", "change") for i in HISTORY_NUMERIC
]
availability_names = [f"lag_{lag}_available" for lag in LAGS]
coverage_names = [f"history_coverage_{window}" for window in WINDOWS]
age_names = ["stock_age"]
difference_names = [f"diff_{lag}_num_{i}" for lag in LAGS for i in HISTORY_NUMERIC]
NUMERIC_NAMES = (
    raw_names + rank_names + lag_names + rolling_names
    + availability_names + coverage_names + age_names + difference_names
)
LEGACY_NUMERIC_PREFIX = 328
NUMERIC_COLUMN_COUNT = 408
assert len(NUMERIC_NAMES) == NUMERIC_COLUMN_COUNT
assert len(NUMERIC_NAMES[:LEGACY_NUMERIC_PREFIX]) == 328


def ensure_directories():
    for directory in (
        OUTPUT_DIR / "common", OUTPUT_DIR / "tree",
        OUTPUT_DIR / "linear", OUTPUT_DIR / "sequence",
    ):
        directory.mkdir(parents=True, exist_ok=True)


def partial_path(final_path: Path) -> Path:
    return final_path.with_name(final_path.stem + ".partial" + final_path.suffix)


def open_atomic_memmap(final_path: Path, dtype, shape):
    temporary = partial_path(final_path)
    if temporary.exists():
        temporary.unlink()
    return temporary, np.lib.format.open_memmap(temporary, mode="w+", dtype=dtype, shape=shape)


def finalize_memmap(temporary: Path, final_path: Path, array):
    array.flush()
    mmap_handle = getattr(array, "_mmap", None)
    if mmap_handle is not None:
        mmap_handle.close()
    del array
    gc.collect()
    if final_path.exists():
        final_path.unlink()
    temporary.replace(final_path)


def stream_sha256(path: Path, block_size: int = 16 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


ensure_directories()
print({"legacy_numeric": 328, "numeric": 408, "tree": 419, "linear": 479})

{'legacy_numeric': 328, 'numeric': 408, 'tree': 419, 'linear': 479}


### 4. 拟合 Train-only 处理状态并构建历史前缀缓存

In [4]:
preprocessing_state = None
prefix_paths = {}


def eligible_mask(time_index: int, require_label: bool) -> np.ndarray:
    mask = np.asarray(data.mask_x[time_index] & data.mask_y[time_index], dtype=bool)
    if require_label:
        mask &= np.isfinite(data.y1[time_index])
    return mask


def deterministic_stocks(stocks: np.ndarray, cap: int = 1200) -> np.ndarray:
    if stocks.size <= cap:
        return stocks
    positions = np.linspace(0, stocks.size - 1, cap, dtype=np.int64)
    return stocks[positions]


def fit_preprocessing_state():
    seen = [set() for _ in range(CATEGORY_FEATURE_COUNT)]
    cat5_max = int(np.max(data.cat_x[:, :, HIGH_CARDINALITY_CATEGORY]))
    cat5_count = np.zeros(cat5_max + 1, dtype=np.int64)
    total = 0
    sample_times = set(np.linspace(TRAIN_START, VALID_START - 1, 64, dtype=int).tolist())
    history_samples = []
    for time_index in range(TRAIN_START, VALID_START):
        stocks = np.flatnonzero(eligible_mask(time_index, True))
        raw = np.asarray(data.cat_x[time_index, stocks], dtype=np.int64)
        for column in range(CATEGORY_FEATURE_COUNT):
            seen[column].update(np.unique(raw[:, column]).tolist())
        cat5 = raw[:, HIGH_CARDINALITY_CATEGORY]
        cat5_count += np.bincount(cat5, minlength=cat5_count.size)
        total += int(cat5.size)
        if time_index in sample_times:
            sampled_stocks = deterministic_stocks(stocks)
            history_samples.append(np.asarray(
                data.num_x[time_index, sampled_stocks][:, HISTORY_NUMERIC], dtype=np.float32
            ))
    mappings = []
    unknown_codes = []
    seen_values = []
    for items in seen:
        ordered = np.asarray(sorted(items), dtype=np.int64)
        maximum = int(max(ordered.max(), 0)) if ordered.size else 0
        mapping = np.full(maximum + 1, -1, dtype=np.int32)
        mapping[ordered] = np.arange(ordered.size, dtype=np.int32)
        mappings.append(mapping)
        unknown_codes.append(int(ordered.size))
        seen_values.append(ordered)
    sampled = np.vstack(history_samples)
    return {
        "mappings": mappings,
        "unknown_codes": unknown_codes,
        "seen_values": seen_values,
        "cat5_count": cat5_count,
        "cat5_total": total,
        "clip_low": np.quantile(sampled, 0.005, axis=0).astype(np.float32),
        "clip_high": np.quantile(sampled, 0.995, axis=0).astype(np.float32),
    }


def build_prefix_cache():
    feature_count = HISTORY_NUMERIC.size
    paths = {
        "cumsum": BUILD_CACHE_DIR / "history_cumsum.dat",
        "cumsq": BUILD_CACHE_DIR / "history_cumsq.dat",
        "cumcount": BUILD_CACHE_DIR / "history_cumcount.dat",
    }
    cumsum = np.memmap(paths["cumsum"], np.float32, "w+", shape=(T + 1, S, feature_count))
    cumsq = np.memmap(paths["cumsq"], np.float32, "w+", shape=(T + 1, S, feature_count))
    cumcount = np.memmap(paths["cumcount"], np.uint16, "w+", shape=(T + 1, S))
    cumsum[0] = 0.0
    cumsq[0] = 0.0
    cumcount[0] = 0
    for time_index in range(T):
        valid = np.asarray(data.mask_x[time_index], dtype=bool)
        values = np.asarray(data.num_x[time_index][:, HISTORY_NUMERIC], dtype=np.float32)
        valid_values = np.where(valid[:, None], values, 0.0)
        cumsum[time_index + 1] = cumsum[time_index] + valid_values
        cumsq[time_index + 1] = cumsq[time_index] + valid_values * valid_values
        cumcount[time_index + 1] = cumcount[time_index] + valid.astype(np.uint16)
        if (time_index + 1) % 300 == 0 or time_index == T - 1:
            print(f"历史前缀缓存 {time_index + 1}/{T}")
    for array in (cumsum, cumsq, cumcount):
        array.flush()
    del cumsum, cumsq, cumcount
    gc.collect()
    return paths


if BUILD_REQUIRED:
    started = time.time()
    preprocessing_state = fit_preprocessing_state()
    prefix_paths = build_prefix_cache()
    print({
        "fit_seconds": round(time.time() - started, 1),
        "unknown_codes": preprocessing_state["unknown_codes"],
        "cat5_total": preprocessing_state["cat5_total"],
    })

### 5. 固化统一样本索引

In [5]:
split_metadata = {}


def write_small_array(final_path: Path, values: np.ndarray):
    temporary, output = open_atomic_memmap(final_path, values.dtype, values.shape)
    output[:] = values
    finalize_memmap(temporary, final_path, output)


def build_common_split(split_name: str, start: int, stop: int, require_label: bool):
    masks = []
    groups = np.empty(stop - start, dtype=np.int32)
    for local_index, time_index in enumerate(range(start, stop)):
        mask = eligible_mask(time_index, require_label)
        masks.append(mask)
        groups[local_index] = int(mask.sum())
    row_count = int(groups.sum())
    if row_count != EXPECTED_ROWS[split_name]:
        raise AssertionError(f"{split_name} 行数 {row_count} != {EXPECTED_ROWS[split_name]}")

    common_dir = OUTPUT_DIR / "common"
    time_tmp, row_time = open_atomic_memmap(common_dir / f"{split_name}_time.npy", np.int32, (row_count,))
    stock_tmp, row_stock = open_atomic_memmap(common_dir / f"{split_name}_stock.npy", np.int32, (row_count,))
    if require_label:
        y_tmp, labels = open_atomic_memmap(common_dir / f"{split_name}_y.npy", np.float32, (row_count,))
        rel_tmp, relevance = open_atomic_memmap(common_dir / f"{split_name}_relevance.npy", np.uint8, (row_count,))
    offset = 0
    for time_index, mask in zip(range(start, stop), masks):
        stocks = np.flatnonzero(mask).astype(np.int32)
        count = stocks.size
        row_time[offset:offset + count] = time_index
        row_stock[offset:offset + count] = stocks
        if require_label:
            current_y = np.asarray(data.y1[time_index, stocks], dtype=np.float32)
            labels[offset:offset + count] = current_y
            relevance[offset:offset + count] = np.minimum(
                np.floor(np.clip(current_y, 0.0, 1.0) * 64.0), 63
            ).astype(np.uint8)
        offset += count
    finalize_memmap(time_tmp, common_dir / f"{split_name}_time.npy", row_time)
    finalize_memmap(stock_tmp, common_dir / f"{split_name}_stock.npy", row_stock)
    if require_label:
        finalize_memmap(y_tmp, common_dir / f"{split_name}_y.npy", labels)
        finalize_memmap(rel_tmp, common_dir / f"{split_name}_relevance.npy", relevance)
    write_small_array(common_dir / f"{split_name}_group_sizes.npy", groups)
    return {"start": start, "stop": stop, "rows": row_count, "time_points": stop - start}


if BUILD_REQUIRED:
    for split_name, (start, stop, require_label) in SPLITS.items():
        split_metadata[split_name] = build_common_split(split_name, start, stop, require_label)
    display(pd.DataFrame(split_metadata).T)

## Results

### 6. 构造408列共享数值块和419列树模型视图

In [6]:
history_cache = None
build_diagnostics = {split: {"lag_missing": {lag: 0 for lag in LAGS}, "lag_filled": {lag: 0 for lag in LAGS}} for split in SPLITS}
unknown_counts = {split: np.zeros(CATEGORY_FEATURE_COUNT, dtype=np.int64) for split in SPLITS}


def open_prefix_cache():
    return {
        "cumsum": np.memmap(prefix_paths["cumsum"], np.float32, "r", shape=(T + 1, S, HISTORY_NUMERIC.size)),
        "cumsq": np.memmap(prefix_paths["cumsq"], np.float32, "r", shape=(T + 1, S, HISTORY_NUMERIC.size)),
        "cumcount": np.memmap(prefix_paths["cumcount"], np.uint16, "r", shape=(T + 1, S)),
    }


def encode_category_codes(raw: np.ndarray) -> np.ndarray:
    encoded = np.empty(raw.shape, dtype=np.int32)
    for column in range(CATEGORY_FEATURE_COUNT):
        mapping = preprocessing_state["mappings"][column]
        unknown = preprocessing_state["unknown_codes"][column]
        values = raw[:, column]
        result = np.full(values.shape, unknown, dtype=np.int32)
        in_range = (values >= 0) & (values < mapping.size)
        mapped = mapping[values[in_range]]
        mapped[mapped < 0] = unknown
        result[in_range] = mapped
        encoded[:, column] = result
    return encoded


def trend_estimate(time_index: int, stocks: np.ndarray) -> np.ndarray:
    start = max(0, time_index - 20)
    values = np.asarray(data.num_x[start:time_index][:, stocks][:, :, HISTORY_NUMERIC], dtype=np.float32)
    valid = np.asarray(data.mask_x[start:time_index][:, stocks], dtype=bool)
    steps = values.shape[0]
    ages = np.arange(steps - 1, -1, -1, dtype=np.float32)
    weights = np.exp(-math.log(2.0) * ages / 5.0)[:, None, None]
    weighted_valid = weights * valid[:, :, None]
    denominator = weighted_valid.sum(axis=0)
    ewma = np.divide(
        (values * weighted_valid).sum(axis=0), denominator,
        out=np.full((stocks.size, HISTORY_NUMERIC.size), np.nan, dtype=np.float32),
        where=denominator > 0,
    )
    last = np.full_like(ewma, np.nan)
    for local_time in range(steps - 1, -1, -1):
        unresolved = ~np.isfinite(last[:, 0]) & valid[local_time]
        if np.any(unresolved):
            last[unresolved] = values[local_time, unresolved]
    if steps >= 2:
        pair_valid = valid[1:] & valid[:-1]
        differences = values[1:] - values[:-1]
        drift = np.divide(
            (differences * pair_valid[:, :, None]).sum(axis=0),
            pair_valid.sum(axis=0)[:, None],
            out=np.zeros_like(ewma), where=pair_valid.sum(axis=0)[:, None] > 0,
        )
    else:
        drift = np.zeros_like(ewma)
    count = valid.sum(axis=0)
    gap = np.zeros(stocks.size, dtype=np.float32)
    for position in range(stocks.size):
        valid_positions = np.flatnonzero(valid[:, position])
        gap[position] = steps - 1 - valid_positions[-1] if valid_positions.size else 20
    reliability = (count / 20.0) * np.exp(-gap / 20.0)
    estimate = 0.5 * last + 0.5 * ewma + 0.5 * reliability[:, None] * drift
    estimate = np.clip(estimate, preprocessing_state["clip_low"], preprocessing_state["clip_high"])
    estimate[count == 0] = np.nan
    return estimate.astype(np.float32)


def build_numeric_block(time_index: int, stocks: np.ndarray, split_name: str) -> np.ndarray:
    count = stocks.size
    current = np.asarray(data.num_x[time_index, stocks][:, SELECTED_NUMERIC], dtype=np.float32)
    current_history = np.asarray(data.num_x[time_index, stocks][:, HISTORY_NUMERIC], dtype=np.float32)

    valid_all = np.asarray(data.mask_x[time_index], dtype=bool)
    all_history = np.asarray(data.num_x[time_index][:, HISTORY_NUMERIC], dtype=np.float32)
    ranks = np.full((S, HISTORY_NUMERIC.size), np.nan, dtype=np.float32)
    valid_count = int(valid_all.sum())
    ranked = rankdata(all_history[valid_all], axis=0, method="average")
    ranks[valid_all] = ((ranked - 1.0) / max(valid_count - 1, 1) - 0.5).astype(np.float32)

    lag_blocks = []
    difference_blocks = []
    lag_availability = []
    trend = None
    for lag in LAGS:
        available_mask = np.asarray(data.mask_x[time_index - lag, stocks], dtype=bool)
        lag_values = np.asarray(data.num_x[time_index - lag, stocks][:, HISTORY_NUMERIC], dtype=np.float32)
        missing_rows = ~available_mask
        missing_count = int(missing_rows.sum())
        build_diagnostics[split_name]["lag_missing"][lag] += missing_count
        if missing_count:
            if trend is None:
                trend = trend_estimate(time_index, stocks)
            lag_values[missing_rows] = trend[missing_rows]
            filled_count = int(np.isfinite(lag_values[missing_rows]).all(axis=1).sum())
            build_diagnostics[split_name]["lag_filled"][lag] += filled_count
        lag_blocks.append(lag_values)
        difference_blocks.append(current_history - lag_values)
        lag_availability.append(available_mask.astype(np.float32))

    rolling_blocks = []
    coverage_columns = []
    for window in WINDOWS:
        window_start = max(0, time_index - window)
        sample_count = (
            history_cache["cumcount"][time_index, stocks]
            - history_cache["cumcount"][window_start, stocks]
        ).astype(np.float32)
        sums = history_cache["cumsum"][time_index, stocks] - history_cache["cumsum"][window_start, stocks]
        squared = history_cache["cumsq"][time_index, stocks] - history_cache["cumsq"][window_start, stocks]
        mean = np.divide(sums, sample_count[:, None], out=np.full_like(sums, np.nan), where=sample_count[:, None] > 0)
        variance = np.divide(squared, sample_count[:, None], out=np.full_like(squared, np.nan), where=sample_count[:, None] > 0) - mean * mean
        std = np.sqrt(np.maximum(variance, 0.0)).astype(np.float32)
        change = np.full_like(mean, np.nan)
        usable = sample_count > 1
        if np.any(usable):
            selected_stocks = stocks[usable]
            first_times = (time_index - sample_count[usable]).astype(np.int32)
            first_values = np.asarray(data.num_x[first_times, selected_stocks][:, HISTORY_NUMERIC], dtype=np.float32)
            last_values = np.asarray(data.num_x[time_index - 1, selected_stocks][:, HISTORY_NUMERIC], dtype=np.float32)
            change[usable] = (last_values - first_values) / (sample_count[usable, None] - 1.0)
        rolling_blocks.extend([mean.astype(np.float32), std, change.astype(np.float32)])
        coverage_columns.append(sample_count / float(window))

    stock_age = np.maximum(time_index - data.first_valid_x[stocks], 0).astype(np.float32)
    legacy = np.column_stack(
        [current, ranks[stocks]] + lag_blocks + rolling_blocks
        + [np.column_stack(lag_availability), np.column_stack(coverage_columns), stock_age[:, None]]
    ).astype(np.float32, copy=False)
    if legacy.shape != (count, LEGACY_NUMERIC_PREFIX):
        raise AssertionError(legacy.shape)
    output = np.column_stack([legacy] + difference_blocks).astype(np.float32, copy=False)
    if output.shape != (count, NUMERIC_COLUMN_COUNT):
        raise AssertionError(output.shape)
    if not np.isfinite(output).all():
        bad = np.argwhere(~np.isfinite(output))[0]
        raise ValueError(f"{split_name} t={time_index} 数值特征非有限：{bad.tolist()}")
    return output


def build_tree_split(split_name: str, start: int, stop: int, require_label: bool):
    row_count = EXPECTED_ROWS[split_name]
    final_path = OUTPUT_DIR / "tree" / f"{split_name}_X.npy"
    temporary, output = open_atomic_memmap(final_path, np.float32, (row_count, 419))
    offset = 0
    for local_index, time_index in enumerate(range(start, stop)):
        stocks = np.flatnonzero(eligible_mask(time_index, require_label))
        count = stocks.size
        numeric = build_numeric_block(time_index, stocks, split_name)
        raw_categories = np.asarray(data.cat_x[time_index, stocks], dtype=np.int64)
        category_codes = encode_category_codes(raw_categories)
        for column in range(CATEGORY_FEATURE_COUNT):
            unknown_counts[split_name][column] += int((category_codes[:, column] == preprocessing_state["unknown_codes"][column]).sum())
        cat5 = raw_categories[:, HIGH_CARDINALITY_CATEGORY]
        counts = np.zeros(count, dtype=np.float32)
        in_range = (cat5 >= 0) & (cat5 < preprocessing_state["cat5_count"].size)
        counts[in_range] = preprocessing_state["cat5_count"][cat5[in_range]]
        frequency = counts / max(preprocessing_state["cat5_total"], 1)
        log_frequency = np.log1p(counts) / max(np.log1p(preprocessing_state["cat5_count"].max()), 1.0)
        block = np.column_stack([numeric, category_codes.astype(np.float32), frequency, log_frequency]).astype(np.float32, copy=False)
        if not np.isfinite(block).all():
            raise ValueError(f"{split_name} t={time_index} tree视图存在非有限值")
        output[offset:offset + count] = block
        offset += count
        if (local_index + 1) % 100 == 0 or time_index == stop - 1:
            print(f"tree/{split_name}: {local_index + 1}/{stop - start}")
    if offset != row_count:
        raise AssertionError(f"{split_name}: {offset} != {row_count}")
    finalize_memmap(temporary, final_path, output)


if BUILD_REQUIRED:
    history_cache = open_prefix_cache()
    for split_name, (start, stop, require_label) in SPLITS.items():
        build_tree_split(split_name, start, stop, require_label)
    display(pd.DataFrame([
        {
            "split": split_name, "lag": lag,
            "历史缺口": build_diagnostics[split_name]["lag_missing"][lag],
            "成功补值": build_diagnostics[split_name]["lag_filled"][lag],
        }
        for split_name in SPLITS for lag in LAGS
    ]))

### 7. 拟合线性视图统计量并生成479列矩阵

In [7]:
linear_state = None


def fit_linear_state():
    train_tree = np.load(OUTPUT_DIR / "tree" / "train_X.npy", mmap_mode="r")
    columns = NUMERIC_COLUMN_COUNT + 2
    total_sum = np.zeros(columns, dtype=np.float64)
    total_squared = np.zeros(columns, dtype=np.float64)
    row_count = train_tree.shape[0]
    chunk_size = 25_000
    for start in range(0, row_count, chunk_size):
        stop = min(start + chunk_size, row_count)
        raw = np.column_stack([train_tree[start:stop, :NUMERIC_COLUMN_COUNT], train_tree[start:stop, 417:419]])
        values = np.asarray(raw, dtype=np.float64)
        total_sum += values.sum(axis=0)
        total_squared += np.square(values).sum(axis=0)
        if stop % 500_000 == 0 or stop == row_count:
            print(f"线性统计量 {stop}/{row_count}")
    mean = total_sum / row_count
    variance = total_squared / row_count - np.square(mean)
    std = np.sqrt(np.maximum(variance, 1e-6))
    del train_tree
    return {"mean": mean.astype(np.float32), "std": std.astype(np.float32)}


def one_hot_low_categories(codes: np.ndarray, offsets: dict[int, int], width: int) -> np.ndarray:
    output = np.zeros((codes.shape[0], width), dtype=np.float32)
    rows = np.arange(codes.shape[0])
    for column in LOW_CARDINALITY_CATEGORIES:
        output[rows, offsets[column] + codes[:, column]] = 1.0
    return output


def build_linear_split(split_name: str):
    tree = np.load(OUTPUT_DIR / "tree" / f"{split_name}_X.npy", mmap_mode="r")
    row_count = tree.shape[0]
    final_path = OUTPUT_DIR / "linear" / f"{split_name}_X.npy"
    temporary, output = open_atomic_memmap(final_path, np.float32, (row_count, 479))
    offsets = {}
    total_one_hot = 0
    for column in LOW_CARDINALITY_CATEGORIES:
        offsets[column] = total_one_hot
        total_one_hot += preprocessing_state["unknown_codes"][column] + 1
    if total_one_hot != 69:
        raise AssertionError(f"One-Hot列数 {total_one_hot} != 69")
    chunk_size = 25_000
    for start in range(0, row_count, chunk_size):
        stop = min(start + chunk_size, row_count)
        numeric = (np.asarray(tree[start:stop, :408]) - linear_state["mean"][:408]) / linear_state["std"][:408]
        codes = np.asarray(tree[start:stop, 408:417], dtype=np.int32)
        one_hot = one_hot_low_categories(codes, offsets, total_one_hot)
        extras = (np.asarray(tree[start:stop, 417:419]) - linear_state["mean"][408:410]) / linear_state["std"][408:410]
        block = np.column_stack([numeric, one_hot, extras]).astype(np.float32, copy=False)
        if not np.isfinite(block).all():
            raise ValueError(f"linear/{split_name}存在非有限值")
        output[start:stop] = block
        if stop % 500_000 == 0 or stop == row_count:
            print(f"linear/{split_name}: {stop}/{row_count}")
    finalize_memmap(temporary, final_path, output)
    del tree


if BUILD_REQUIRED:
    linear_state = fit_linear_state()
    for split_name in SPLITS:
        build_linear_split(split_name)
    print({"linear_mean_shape": linear_state["mean"].shape, "linear_std_min": float(linear_state["std"].min())})

### 8. 生成TCN固定40列时序视图

In [8]:
def build_sequence_view():
    x_path = OUTPUT_DIR / "sequence" / "X.npy"
    mask_path = OUTPUT_DIR / "sequence" / "mask_x.npy"
    x_tmp, sequence = open_atomic_memmap(x_path, np.float32, (T, S, 40))
    mask_tmp, sequence_mask = open_atomic_memmap(mask_path, np.bool_, (T, S))
    mean = linear_state["mean"][:40]
    std = linear_state["std"][:40]
    for time_index in range(T):
        valid = np.asarray(data.mask_x[time_index], dtype=bool)
        values = np.asarray(data.num_x[time_index][:, SELECTED_NUMERIC], dtype=np.float32)
        standardized = (values - mean) / std
        standardized[~valid] = 0.0
        if not np.isfinite(standardized).all():
            raise ValueError(f"sequence t={time_index}存在非有限值")
        sequence[time_index] = standardized
        sequence_mask[time_index] = valid
        if (time_index + 1) % 300 == 0 or time_index == T - 1:
            print(f"sequence: {time_index + 1}/{T}")
    finalize_memmap(x_tmp, x_path, sequence)
    finalize_memmap(mask_tmp, mask_path, sequence_mask)


if BUILD_REQUIRED:
    build_sequence_view()

## Checks

### 9. 全量验收、旧模型兼容和版本清单

In [9]:
validation_summary = {}
compatibility_summary = {"status": "not_run"}


def scan_finite(array: np.ndarray, row_chunk: int = 100_000):
    if array.ndim == 3:
        for start in range(0, array.shape[0], 32):
            if not np.isfinite(array[start:start + 32]).all():
                return False
    else:
        for start in range(0, array.shape[0], row_chunk):
            if not np.isfinite(array[start:start + row_chunk]).all():
                return False
    return True


def validate_materialized_outputs():
    checks = {}
    for split_name, rows in EXPECTED_ROWS.items():
        tree = np.load(OUTPUT_DIR / "tree" / f"{split_name}_X.npy", mmap_mode="r")
        linear = np.load(OUTPUT_DIR / "linear" / f"{split_name}_X.npy", mmap_mode="r")
        row_time = np.load(OUTPUT_DIR / "common" / f"{split_name}_time.npy", mmap_mode="r")
        row_stock = np.load(OUTPUT_DIR / "common" / f"{split_name}_stock.npy", mmap_mode="r")
        groups = np.load(OUTPUT_DIR / "common" / f"{split_name}_group_sizes.npy", mmap_mode="r")
        assert tree.shape == (rows, 419)
        assert linear.shape == (rows, 479)
        assert row_time.shape == row_stock.shape == (rows,)
        assert int(groups.sum()) == rows
        assert scan_finite(tree) and scan_finite(linear)
        offsets = []
        current = 408
        for column in LOW_CARDINALITY_CATEGORIES:
            width = preprocessing_state["unknown_codes"][column] + 1
            offsets.append((current, current + width))
            current += width
        for start in range(0, rows, 100_000):
            block = linear[start:start + 100_000]
            for left, right in offsets:
                if not np.allclose(block[:, left:right].sum(axis=1), 1.0):
                    raise AssertionError(f"{split_name} One-Hot组异常 {left}:{right}")
        checks[split_name] = {"rows": rows, "tree_finite": True, "linear_finite": True}

    sequence = np.load(OUTPUT_DIR / "sequence" / "X.npy", mmap_mode="r")
    sequence_mask = np.load(OUTPUT_DIR / "sequence" / "mask_x.npy", mmap_mode="r")
    assert sequence.shape == (T, S, 40) and sequence_mask.shape == (T, S)
    assert scan_finite(sequence)
    assert np.array_equal(sequence_mask, np.asarray(data.mask_x))

    test_time = np.load(OUTPUT_DIR / "common" / "test_time.npy", mmap_mode="r")
    test_stock = np.load(OUTPUT_DIR / "common" / "test_stock.npy", mmap_mode="r")
    reconstructed = np.zeros((T - TEST_START, S), dtype=bool)
    reconstructed[test_time - TEST_START, test_stock] = True
    official = np.asarray(data.mask_x[TEST_START:T] & data.mask_y[TEST_START:T], dtype=bool)
    assert np.array_equal(reconstructed, official)

    assert build_diagnostics["train"]["lag_missing"][60] == 1424
    assert build_diagnostics["train"]["lag_missing"][60] == build_diagnostics["train"]["lag_filled"][60]
    at_1288 = np.flatnonzero(eligible_mask(1288, True))
    missing_1288 = int((~np.asarray(data.mask_x[1288 - 60, at_1288], dtype=bool)).sum())
    assert missing_1288 == 17
    checks["sequence"] = {"shape": list(sequence.shape), "finite": True}
    checks["test_mask"] = {"count": int(reconstructed.sum()), "matches_official": True}
    checks["lag60"] = {"train_missing": 1424, "filled": 1424, "time_1288_missing": 17}
    return checks


def legacy_model_compatibility():
    legacy_dir = ROOT_DIR / "03_模型训练" / "y1_pipeline_v2"
    model_path = legacy_dir / "outputs" / "lgbm_model_v2.txt"
    if not model_path.exists():
        return {"status": "skipped", "reason": "旧LightGBM模型不存在"}
    try:
        import lightgbm as lgb
        temporary_dir = Path(tempfile.gettempdir()) / "y1_processed_compat"
        temporary_dir.mkdir(parents=True, exist_ok=True)
        ascii_model = temporary_dir / "model.txt"
        shutil.copyfile(model_path, ascii_model)
        booster = lgb.Booster(model_file=str(ascii_model))
        results = {"status": "passed", "feature_count": 328}
        for split_name, split_start in (("valid", VALID_START), ("test", TEST_START)):
            tree = np.load(OUTPUT_DIR / "tree" / f"{split_name}_X.npy", mmap_mode="r")
            row_time = np.load(OUTPUT_DIR / "common" / f"{split_name}_time.npy", mmap_mode="r")
            row_stock = np.load(OUTPUT_DIR / "common" / f"{split_name}_stock.npy", mmap_mode="r")
            expected_grid = np.load(legacy_dir / "outputs" / f"{split_name}_lgbm.npy", mmap_mode="r")
            expected = expected_grid[row_time - split_start, row_stock]
            maximum_error = 0.0
            chunk_size = 100_000
            for start in range(0, tree.shape[0], chunk_size):
                stop = min(start + chunk_size, tree.shape[0])
                predicted = booster.predict(np.asarray(tree[start:stop, :328]), num_iteration=booster.num_trees())
                maximum_error = max(maximum_error, float(np.max(np.abs(predicted - expected[start:stop]))))
            if maximum_error > 1e-6:
                raise AssertionError(f"{split_name}旧模型预测最大误差 {maximum_error}")
            results[f"{split_name}_max_abs_error"] = maximum_error
        if ascii_model.exists():
            ascii_model.unlink()
        return results
    except Exception as error:
        return {"status": "failed", "error": repr(error)}


def json_ready(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    return value


if BUILD_REQUIRED:
    validation_summary = validate_materialized_outputs()
    compatibility_summary = legacy_model_compatibility()
    if compatibility_summary.get("status") != "passed":
        raise RuntimeError(f"旧328列兼容验收未通过：{compatibility_summary}")

    file_records = {}
    output_files = sorted(
        path for path in OUTPUT_DIR.rglob("*.npy") if ".partial" not in path.name
    )
    for index, path in enumerate(output_files, start=1):
        relative = path.relative_to(OUTPUT_DIR).as_posix()
        file_records[relative] = {
            "bytes": path.stat().st_size,
            "sha256": stream_sha256(path),
        }
        print(f"哈希 {index}/{len(output_files)}: {relative}")

    manifest = {
        "dataset_version": "processed_data_v1",
        "status": "ready",
        "source": {"path": "data.z", "bytes": SOURCE_PATH.stat().st_size, "sha256": stream_sha256(SOURCE_PATH)},
        "dimensions": {"time": T, "stock": S, "raw_numeric": 99, "raw_category": 9},
        "splits": split_metadata,
        "expected_rows": EXPECTED_ROWS,
        "fit_interval": [TRAIN_START, VALID_START],
        "selected_numeric_features": SELECTED_NUMERIC,
        "selected_history_features": HISTORY_NUMERIC,
        "features": {
            "numeric_names": NUMERIC_NAMES,
            "legacy_numeric_prefix": LEGACY_NUMERIC_PREFIX,
            "numeric_count": 408,
            "tree_count": 419,
            "tree_categorical_indices": list(range(408, 417)),
            "linear_count": 479,
            "linear_one_hot_range": [408, 477],
            "sequence_count": 40,
        },
        "processing": {
            "observed_value_winsorization": False,
            "trend_imputation": "last + EWMA + reliability-weighted drift; train 0.5%-99.5% guardrail",
            "lags": LAGS,
            "rolling_windows": WINDOWS,
            "cat5_target_encoding": False,
            "unknown_bucket": True,
        },
        "category_state": {
            "seen_values": preprocessing_state["seen_values"],
            "unknown_codes": preprocessing_state["unknown_codes"],
            "cat5_total": preprocessing_state["cat5_total"],
            "cat5_count": preprocessing_state["cat5_count"],
        },
        "linear_state": linear_state,
        "trend_clip": {"low": preprocessing_state["clip_low"], "high": preprocessing_state["clip_high"]},
        "diagnostics": {"build": build_diagnostics, "unknown_counts": unknown_counts},
        "validation": validation_summary,
        "legacy_compatibility": compatibility_summary,
        "files": file_records,
    }
    manifest = json_ready(manifest)
    manifest_tmp = MANIFEST_PATH.with_suffix(".json.partial")
    manifest_tmp.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    if MANIFEST_PATH.exists():
        MANIFEST_PATH.unlink()
    manifest_tmp.replace(MANIFEST_PATH)
    ready_payload = {
        "dataset_version": "processed_data_v1",
        "manifest_sha256": stream_sha256(MANIFEST_PATH),
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    ready_tmp = READY_PATH.with_suffix(".partial")
    ready_tmp.write_text(json.dumps(ready_payload, ensure_ascii=False, indent=2), encoding="utf-8")
    ready_tmp.replace(READY_PATH)
    print("READY 已生成")

    del history_cache, data
    gc.collect()
    if BUILD_CACHE_DIR.exists():
        shutil.rmtree(BUILD_CACHE_DIR)
        print("临时构建缓存已清理")

### 10. READY复核与交付摘要

In [10]:
if not READY_PATH.exists() or not MANIFEST_PATH.exists():
    raise RuntimeError("processed_data_v1 未完成：READY或manifest缺失")
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
ready = json.loads(READY_PATH.read_text(encoding="utf-8"))
if stream_sha256(MANIFEST_PATH) != ready["manifest_sha256"]:
    raise RuntimeError("manifest哈希与READY不一致")
for relative, record in manifest["files"].items():
    path = OUTPUT_DIR / relative
    if not path.exists() or path.stat().st_size != record["bytes"]:
        raise RuntimeError(f"产出文件缺失或大小异常：{relative}")

total_bytes = sum(record["bytes"] for record in manifest["files"].values())
delivery_summary = pd.DataFrame([
    {"视图": "Tree", "列数": 419, "用途": "LightGBM/树模型"},
    {"视图": "Linear", "列数": 479, "用途": "线性回归/SGD"},
    {"视图": "Sequence", "列数": 40, "用途": "TCN 486期窗口"},
])
display(delivery_summary)
print({
    "status": manifest["status"],
    "rows": manifest["expected_rows"],
    "total_size_gib": round(total_bytes / 2**30, 3),
    "legacy_compatibility": manifest["legacy_compatibility"],
    "ready": str(READY_PATH),
})

,视图,列数,用途
0,Tree,419,LightGBM/树模型
1,Linear,479,线性回归/SGD
2,Sequence,40,TCN 486期窗口


{'status': 'ready', 'rows': {'train': 6489099, 'valid': 982972, 'test': 2042538}, 'total_size_gib': 34.789, 'legacy_compatibility': {'status': 'passed', 'feature_count': 328, 'valid_max_abs_error': 5.957341553397555e-08, 'test_max_abs_error': 5.960239524149813e-08}, 'ready': 'D:\\google_dl\\book\\友安杯\\02_数据处理\\processed_data_v1\\READY'}


## Takeaways

- `processed_data_v1/READY` 存在后，模型训练只能读取这里的固定视图。
- `tree` 前328列与旧LightGBM数值输入兼容；第329～408列是新增直接差分。
- `linear` 已完成Train-only标准化和类别One-Hot；`sequence` 已完成标准化并保留独立掩码。
- 当前最终提交文件未被本 Notebook 修改。